# BBVA Recomendador — Pipeline Completo


## 1. Imports y Carga de Datos

In [1]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import matplotlib.pyplot as plt
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error

train    = pd.read_csv('01dataBaseTrainTrxRec.csv')
perfil   = pd.read_csv('02dataBasePerfilRec.csv')
test_key = pd.read_csv('05dataBaseTestKeyRec.csv')
test     = pd.read_csv('03dataBaseTestRec.csv')

print('Train:', train.shape)
print('Perfil:', perfil.shape)
print('Test key:', test_key.shape)

Train: (1591617, 8)
Perfil: (30000, 14)
Test key: (467203, 2)


## 2. Cleaning — Train

In [2]:
# Fechas y features temporales
train['fechaOper']   = pd.to_datetime(train['fechaOper'])
train['mes']         = train['fechaOper'].dt.month
train['dia_semana']  = train['fechaOper'].dt.dayofweek
train['es_finde']    = train['dia_semana'].isin([5, 6]).astype(int)
train['es_diciembre']= (train['mes'] == 12).astype(int)
train['es_julio']    = (train['mes'] == 7).astype(int)

# Flag de recencia (últimos 3 meses)
fecha_max = train['fechaOper'].max()
train['es_reciente'] = (train['fechaOper'] >= fecha_max - pd.DateOffset(months=3)).astype(int)

# Nulos
train['codGiro']    = train['codGiro'].fillna(0).astype(int)
train['ubigeoEstab']= train['ubigeoEstab'].fillna(0).astype(int)

# Target
train['target'] = np.log1p(train['ratingMonto'])

print('Train shape:', train.shape)

Train shape: (1591617, 15)


## 3. Cleaning — Perfil

In [3]:
# Nulos en saldos = sin tarjeta en esa entidad
for col in ['saldoTcEntidad1','saldoTcEntidad2','saldoTcEntidad3','saldoTcEntidad4']:
    perfil[col] = perfil[col].fillna('SinSaldo')

# Rangos con nulos: rellenar con moda
for col in ['rangoIngreso', 'rangoEdad']:
    perfil[col] = perfil[col].fillna(perfil[col].mode()[0])
perfil['ubigeoCliente'] = perfil['ubigeoCliente'].fillna(perfil['ubigeoCliente'].mode()[0])

# Encoding ordinal correcto
rango_map = {'Rango1':1,'Rango2':2,'Rango3':3,'Rango4':4,'Rango5':5,'Rango6':6}
saldo_map = {'SinSaldo':0,'Rango1':1,'Rango2':2,'Rango3':3,'Rango4':4,'Rango5':5,'Rango6':6}

for col in ['rangoEdad','rangoIngreso','rangoCtdProdAct','rangoCtdProdPas','rangoCtdProdSeg']:
    perfil[col] = perfil[col].map(rango_map)
for col in ['saldoTcEntidad1','saldoTcEntidad2','saldoTcEntidad3','saldoTcEntidad4']:
    perfil[col] = perfil[col].map(saldo_map)

print('Perfil shape:', perfil.shape)

Perfil shape: (30000, 14)


## 4. Merge Train + Perfil

In [4]:
train = train.merge(perfil, on='codCliente', how='left')
print('Train limpio:', train.shape)
print('Nulos en train:', train.isnull().sum().sum())

Train limpio: (1591617, 28)
Nulos en train: 0


## 5. Feature Engineering

In [5]:
# 5.1 Features por cliente
client_feats = train.groupby('codCliente').agg(
    total_trx_cliente     = ('ctdTrx', 'sum'),
    avg_trx_cliente       = ('ctdTrx', 'mean'),
    total_estab_visitados = ('codEstab', 'nunique'),
    total_giros_visitados = ('codGiro', 'nunique'),
    avg_rating_cliente    = ('ratingMonto', 'mean'),
    max_rating_cliente    = ('ratingMonto', 'max'),
    std_rating_cliente    = ('ratingMonto', 'std'),
    pct_lima_estab_cliente= ('flagLimaProvEstab', 'mean'),
    meses_activo          = ('mes', 'nunique'),
    pct_finde_cliente     = ('es_finde', 'mean'),
).reset_index()

In [6]:
# 5.2 Features por establecimiento
estab_feats = train.groupby('codEstab').agg(
    total_clientes_estab = ('codCliente', 'nunique'),
    total_trx_estab      = ('ctdTrx', 'sum'),
    avg_trx_estab        = ('ctdTrx', 'mean'),
    avg_rating_estab     = ('ratingMonto', 'mean'),
    max_rating_estab     = ('ratingMonto', 'max'),
    std_rating_estab     = ('ratingMonto', 'std'),
    pct_finde_estab      = ('es_finde', 'mean'),
    meses_activo_estab   = ('mes', 'nunique'),
).reset_index()

In [7]:
# 5.3 Features por giro
giro_feats = train.groupby('codGiro').agg(
    avg_rating_giro    = ('ratingMonto', 'mean'),
    total_clientes_giro= ('codCliente', 'nunique'),
    total_trx_giro     = ('ctdTrx', 'sum'),
    popularidad_giro   = ('codEstab', 'nunique'),
).reset_index()

# Giro favorito por cliente
client_giro_feats = train.groupby(['codCliente','codGiro']).agg(
    avg_rating_cliente_giro = ('ratingMonto', 'mean'),
).reset_index()
giro_favorito = client_giro_feats.loc[
    client_giro_feats.groupby('codCliente')['avg_rating_cliente_giro'].idxmax()
][['codCliente','codGiro']].rename(columns={'codGiro':'giro_favorito'})
client_feats = client_feats.merge(giro_favorito, on='codCliente', how='left')

In [8]:
# 5.4 Features por par cliente-establecimiento
pair_feats = train.groupby(['codCliente','codEstab']).agg(
    trx_par          = ('ctdTrx', 'sum'),
    avg_rating_par   = ('ratingMonto', 'mean'),
    max_rating_par   = ('ratingMonto', 'max'),
    visitas_par      = ('ratingMonto', 'count'),
    meses_visitado_par= ('mes', 'nunique'),
    pct_finde_par    = ('es_finde', 'mean'),
).reset_index()

In [9]:
# 5.5 Cold start: giro por establecimiento
estab_giro  = train[['codEstab','codGiro']].dropna().drop_duplicates('codEstab')
estab_feats = estab_feats.merge(estab_giro, on='codEstab', how='left')
estab_feats = estab_feats.merge(
    giro_feats[['codGiro','avg_rating_giro','total_clientes_giro']],
    on='codGiro', how='left'
)

In [10]:
# 5.6 Target encoding — calculado ANTES de build_dataset
# se guarda el codGiro y ubigeoEstab del train original (antes de merges)
giro_target = train.groupby('codGiro')['ratingMonto'].mean().reset_index()
giro_target.columns = ['codGiro', 'target_enc_giro']

ubigeo_target = train.groupby('ubigeoEstab')['ratingMonto'].mean().reset_index()
ubigeo_target.columns = ['ubigeoEstab', 'target_enc_ubigeo']

In [11]:
# 5.7 Recencia: rating de los últimos 3 meses por par
pair_reciente = train[train['es_reciente']==1].groupby(['codCliente','codEstab']).agg(
    avg_rating_par_reciente = ('ratingMonto', 'mean'),
    trx_par_reciente        = ('ctdTrx', 'sum'),
).reset_index()

In [12]:
# 5.8 Tendencia: primera vs segunda mitad del historial
train_sorted = train.sort_values(['codCliente','codEstab','fechaOper'])
train_sorted['cumidx']  = train_sorted.groupby(['codCliente','codEstab']).cumcount()
train_sorted['total_v'] = train_sorted.groupby(['codCliente','codEstab'])['codEstab'].transform('count')
train_sorted['es_segunda_mitad'] = (train_sorted['cumidx'] >= train_sorted['total_v']/2).astype(int)

tendencia = train_sorted.groupby(['codCliente','codEstab','es_segunda_mitad'])['ratingMonto'].mean().unstack()
tendencia.columns = ['rating_primera_mitad','rating_segunda_mitad']
tendencia['tendencia_rating'] = tendencia['rating_segunda_mitad'] - tendencia['rating_primera_mitad']
tendencia = tendencia.reset_index()[['codCliente','codEstab','tendencia_rating']]

## 6. build_dataset

In [13]:
GLOBAL_AVG = 0.0132

def build_dataset(df, client_feats, estab_feats, pair_feats):
    df = df.copy()

    # Target encoding ANTES de cualquier merge que pueda renombrar columnas
    df = df.merge(giro_target,   on='codGiro',    how='left')
    df = df.merge(ubigeo_target, on='ubigeoEstab', how='left')

    # Merges principales
    df = df.merge(client_feats, on='codCliente',              how='left')
    df = df.merge(estab_feats,  on='codEstab',                how='left')
    df = df.merge(pair_feats,   on=['codCliente','codEstab'], how='left')

    # Merges adicionales
    df = df.merge(pair_reciente, on=['codCliente','codEstab'], how='left')
    df = df.merge(tendencia,     on=['codCliente','codEstab'], how='left')

    # ── Fillna pares sin historial directo ────────────────────
    df['trx_par']        = df['trx_par'].fillna(0)
    df['avg_rating_par'] = df['avg_rating_par'].fillna(df['avg_rating_estab'])
    df['max_rating_par'] = df['max_rating_par'].fillna(df['avg_rating_estab'])
    df['visitas_par']    = df['visitas_par'].fillna(0)

    # ── Fillna establecimientos sin historial (cold start) ────
    df['avg_rating_estab']     = df['avg_rating_estab'].fillna(df['avg_rating_giro'])
    df['max_rating_estab']     = df['max_rating_estab'].fillna(df['avg_rating_giro'])
    df['total_clientes_estab'] = df['total_clientes_estab'].fillna(0)
    df['total_trx_estab']      = df['total_trx_estab'].fillna(0)
    df['avg_trx_estab']        = df['avg_trx_estab'].fillna(0)
    df['pct_finde_estab']      = df['pct_finde_estab'].fillna(0)
    df['meses_activo_estab']   = df['meses_activo_estab'].fillna(0)
    df['std_rating_estab']     = df['std_rating_estab'].fillna(0)

    # ── Fallback global para giro desconocido ─────────────────
    df['avg_rating_giro']     = df['avg_rating_giro'].fillna(GLOBAL_AVG)
    df['total_clientes_giro'] = df['total_clientes_giro'].fillna(0)
    df['avg_rating_estab']    = df['avg_rating_estab'].fillna(GLOBAL_AVG)
    df['max_rating_estab']    = df['max_rating_estab'].fillna(GLOBAL_AVG)

    # ── Fillna par sin historial temporal ─────────────────────
    df['meses_visitado_par'] = df['meses_visitado_par'].fillna(0)
    df['pct_finde_par']      = df['pct_finde_par'].fillna(df['pct_finde_estab'])
    df['std_rating_cliente'] = df['std_rating_cliente'].fillna(0)

    # ── Fillna segunda pasada para avg/max_rating_par ─────────
    df['avg_rating_par'] = df['avg_rating_par'].fillna(GLOBAL_AVG)
    df['max_rating_par'] = df['max_rating_par'].fillna(GLOBAL_AVG)

    # ── Fillna nuevas features ────────────────────────────────
    df['avg_rating_par_reciente'] = df['avg_rating_par_reciente'].fillna(0)
    df['trx_par_reciente']        = df['trx_par_reciente'].fillna(0)
    df['tendencia_rating']        = df['tendencia_rating'].fillna(0)
    df['target_enc_giro']         = df['target_enc_giro'].fillna(GLOBAL_AVG)
    df['target_enc_ubigeo']       = df['target_enc_ubigeo'].fillna(GLOBAL_AVG)

    # ── Ratios e interacciones ────────────────────────────────
    df['ratio_trx_cliente_estab']       = df['trx_par'] / (df['total_trx_cliente'] + 1)
    df['ratio_estab_sobre_cliente']     = df['total_clientes_estab'] / (df['total_estab_visitados'] + 1)
    df['ratio_estab_sobre_cliente_log'] = np.log1p(df['ratio_estab_sobre_cliente'])
    df['diff_rating_par_vs_cliente']    = df['avg_rating_par'] - df['avg_rating_cliente']
    df['diff_rating_par_vs_estab']      = df['avg_rating_par'] - df['avg_rating_estab']

    return df

## 7. Construir X_train y X_test

In [16]:
X_train = build_dataset(train, client_feats, estab_feats, pair_feats)

# test_key: merge con perfil primero
test_key = test_key.merge(perfil, on='codCliente', how='left')
#X_test   = build_dataset(test_key, client_feats, estab_feats, pair_feats)

print('X_train shape:', X_train.shape)
#print('X_test shape: ', X_test.shape)

X_train shape: (1591617, 66)


In [18]:
# Definir features
drop_cols = [
    'codCliente', 'codEstab',
    'ratingMonto', 'target',
    'fechaOper', 'mes', 'dia_semana', 'es_finde', 'es_diciembre', 'es_julio',
    'es_reciente',
    'codGiro_x', 'codGiro_y', 'codGiro',
    'ubigeoEstab_x', 'ubigeoEstab_y', 'ubigeoEstab',
    'flagLimaProvEstab',
]

features = [c for c in X_train.columns if c not in drop_cols]
features = [c for c in features if c in X_train.columns]

y_train = X_train['target']

print(f'Features totales: {len(features)}')
print(f'Nulos en X_train: {X_train[features].isnull().sum().sum()}')
#print(f'Nulos en X_test:  {X_test[features].isnull().sum().sum()}')
print('\nLista de features:')
for f in features:
    print(f'  - {f}')

Features totales: 51
Nulos en X_train: 0

Lista de features:
  - ctdTrx
  - rangoEdad
  - rangoIngreso
  - flagGenero
  - flagLimaProvCliente
  - ubigeoCliente
  - rangoCtdProdAct
  - rangoCtdProdPas
  - rangoCtdProdSeg
  - flagBxi
  - saldoTcEntidad1
  - saldoTcEntidad2
  - saldoTcEntidad3
  - saldoTcEntidad4
  - target_enc_giro
  - target_enc_ubigeo
  - total_trx_cliente
  - avg_trx_cliente
  - total_estab_visitados
  - total_giros_visitados
  - avg_rating_cliente
  - max_rating_cliente
  - std_rating_cliente
  - pct_lima_estab_cliente
  - meses_activo
  - pct_finde_cliente
  - giro_favorito
  - total_clientes_estab
  - total_trx_estab
  - avg_trx_estab
  - avg_rating_estab
  - max_rating_estab
  - std_rating_estab
  - pct_finde_estab
  - meses_activo_estab
  - avg_rating_giro
  - total_clientes_giro
  - trx_par
  - avg_rating_par
  - max_rating_par
  - visitas_par
  - meses_visitado_par
  - pct_finde_par
  - avg_rating_par_reciente
  - trx_par_reciente
  - tendencia_rating
  - ratio

## 8. Modelo — LightGBM con KFold

In [19]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)
oof_preds       = np.zeros(len(X_train))
feat_importance = np.zeros(len(features))

params = {
    'n_estimators'     : 500,
    'learning_rate'    : 0.03,
    'num_leaves'       : 63,
    'min_child_samples': 20,
    'subsample'        : 0.8,
    'colsample_bytree' : 0.8,
    'reg_alpha'        : 0.1,
    'reg_lambda'       : 0.1,
    'min_gain_to_split': 0.01,
    'random_state'     : 42,
    'n_jobs'           : -1,
    'verbose'          : -1,
}

for fold, (tr_idx, val_idx) in enumerate(kf.split(X_train)):
    print(f'\n===== Fold {fold+1} =====')
    X_tr  = X_train[features].iloc[tr_idx]
    X_val = X_train[features].iloc[val_idx]
    y_tr  = y_train.iloc[tr_idx]
    y_val = y_train.iloc[val_idx]

    model = lgb.LGBMRegressor(**params)
    model.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(100)]
    )
    oof_preds[val_idx]  = model.predict(X_val)
    feat_importance    += model.feature_importances_ / 5

oof_rmse      = np.sqrt(mean_squared_error(y_train, oof_preds))
oof_real      = np.expm1(oof_preds)
y_real        = np.expm1(y_train)
oof_rmse_real = np.sqrt(mean_squared_error(y_real, oof_real))

print(f'\n✅ OOF RMSE (log scale):     {oof_rmse:.6f}')
print(f'✅ OOF RMSE (escala original): {oof_rmse_real:.6f}')


===== Fold 1 =====
[100]	valid_0's l2: 7.61067e-05
[200]	valid_0's l2: 7.35213e-05

===== Fold 2 =====
[100]	valid_0's l2: 8.18847e-05


KeyboardInterrupt: 

In [ ]:
# Feature importance
fi_df = pd.DataFrame({'feature': features, 'importance': feat_importance})
fi_df = fi_df.sort_values('importance', ascending=False)

print('Top 15 features más importantes:')
print(fi_df.head(15).to_string(index=False))

fi_df.head(20).plot(kind='barh', x='feature', y='importance', figsize=(10,8),
                    color='steelblue', legend=False)
plt.title('Feature Importance — LightGBM')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

## 9. Submission

In [ ]:
# Reentrenar con todos los datos
print('Entrenando modelo final con todo el train...')
model_final = lgb.LGBMRegressor(**params)
model_final.fit(
    X_train[features], y_train,
    callbacks=[lgb.log_evaluation(100)]
)

# Predecir
test_preds = model_final.predict(X_test[features])
test_preds = np.expm1(test_preds)
test_preds = np.clip(test_preds, 0, 1)

# Guardar
test['ratingMonto'] = test_preds
test.to_csv('submission.csv', index=False)

print(f'\n✅ Submission generado: {len(test)} filas')
print(test.head(10))